# Module 5: Subqueries, CTEs, and Set Operations

**ALY 6420 | Building Derived Datasets and Multi-Step Queries**

*Course Lecture Notes*


## Module 5

### Building queries from other queries

Module 4 focused on combining related tables with `JOIN`. Module 5 adds another family of tools for assembling analytical datasets:

- **subqueries** place one query inside another,
- **common table expressions (CTEs)** name intermediate query results with `WITH`, and
- **set operations** combine complete result sets vertically.

These tools overlap. The goal is not to memorize one “best” syntax, but to learn how each form expresses a different analytical intent and how to choose a structure that is correct, readable, and easy to validate.


## Learning objectives

By the end of this lecture, you should be able to:

- write non-correlated subqueries in `WHERE` and `FROM`
- recognize scalar, list-returning, and table-returning subqueries
- write correlated subqueries and explain how they depend on the outer query
- use `EXISTS` and `NOT EXISTS` to test whether related records are present
- write CTEs with `WITH` and chain several CTEs into a readable sequence
- use `UNION`, `UNION ALL`, `INTERSECT`, and `EXCEPT`
- explain column-count, type-compatibility, duplicate-handling, and ordering rules for set operations
- choose among a `JOIN`, subquery, CTE, and set operation based on the analytical question


## Required preparation

### Principal resource

- Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for Data Analytics* (4th ed.), **Chapter 7: Defining Datasets from Existing Datasets**.

Chapter 7 develops the idea that the output of a `SELECT` query is itself a relation that can be reused as input to another query. It then introduces subqueries, CTEs, JOINs, and set operations as ways to construct more useful datasets from existing data.

### Practice environment

Run the examples against the **Pagila** PostgreSQL database in DBeaver. Predict what each query should return before executing it, then inspect both the row count and the result grain.


# Part 1: The Big Idea — A Query Result Can Become Input


## From stored tables to derived datasets

A table is not the only possible source for a query. A `SELECT` statement also produces a two-dimensional result: rows and columns.

That result can be treated as another dataset.

```text
stored tables
     |
     v
 SELECT query
     |
     v
derived result
     |
     v
another query can use that result
```

This idea is the foundation of subqueries and CTEs.


## A simple business question in two steps

Suppose a manager asks:

> Which films have a rental rate above the average rental rate?

There are really two calculations hidden inside the sentence:

1. compute the average rental rate,
2. keep films whose rental rate is greater than that average.

SQL lets the first query become an input to the second.


## Manual two-query approach

First compute the average:

```sql
SELECT AVG(rental_rate) AS avg_rental_rate
FROM film;
```

Imagine the result is one number. You could copy that number into a second query:

```sql
SELECT title, rental_rate
FROM film
WHERE rental_rate > 2.98;
```

This works, but it is manual and brittle. If the data changes, the copied constant may be wrong.


## Let the database connect the steps

Use a subquery so the average is computed at execution time:

```sql
SELECT title, rental_rate
FROM film
WHERE rental_rate > (
    SELECT AVG(rental_rate)
    FROM film
)
ORDER BY rental_rate DESC, title;
```

Now the query always compares each film with the current average.


## Analytical habit: identify the hidden steps

Before writing SQL, translate the business question into intermediate results.

For example:

> Which customers spent more than the average customer?

Possible steps:

```text
payment rows
   ↓
sum payments per customer
   ↓
calculate the average customer total
   ↓
keep customers above that average
   ↓
attach customer names
```

Once the steps are visible, the SQL structure becomes much easier to choose.


# Part 2: Subqueries — One Query Inside Another


## What is a subquery?

A **subquery** is a complete `SELECT` statement embedded inside another SQL statement.

The outer query is often called the **main query**.

Basic pattern:

```sql
SELECT ...
FROM ...
WHERE column_operator (
    SELECT ...
    FROM ...
);
```

The shape of the inner result must match the place where the subquery is used.


## Three useful result shapes

A subquery can return different shapes.

| Shape | Example result | Common use |
|---|---|---|
| Scalar | one value | comparison such as `>` or `=` |
| Single column | many values | membership test with `IN` |
| Table | rows and columns | source in `FROM` |

A useful question is therefore:

> **What shape does the outer query need from the inner query?**


## Scalar subquery

A scalar subquery returns exactly one value.

```sql
SELECT title, length
FROM film
WHERE length > (
    SELECT AVG(length)
    FROM film
)
ORDER BY length DESC;
```

The inner query produces one average. The outer query compares each film's `length` to that one number.


## Why aggregate functions often appear in scalar subqueries

Aggregate functions such as:

- `AVG()`
- `MIN()`
- `MAX()`
- `SUM()`
- `COUNT()`

can reduce many rows to one value.

That makes them natural inputs to scalar comparisons.

```sql
WHERE rental_rate > (SELECT AVG(rental_rate) FROM film)
```


## Quick check: will this work?

```sql
SELECT title
FROM film
WHERE rental_rate = (
    SELECT rental_rate
    FROM film
);
```

The inner query returns **many rows**, not one scalar value.

A comparison using `=` expects a single value. PostgreSQL will reject a scalar subquery that returns more than one row.

> **Rule:** match the operator to the shape of the subquery result.


## Single-column subquery with `IN`

When the inner query returns a list of values, `IN` is often appropriate.

```sql
SELECT customer_id, first_name, last_name
FROM customer
WHERE customer_id IN (
    SELECT customer_id
    FROM payment
);
```

The inner query produces customer IDs. The outer query keeps customers whose ID appears in that result.


## Simple real-world interpretation

The pattern:

```sql
WHERE key IN (
    SELECT key
    FROM related_table
    WHERE condition
)
```

means:

> Keep rows whose key belongs to a computed set.

Examples include:

- employees in departments with open positions,
- products sold in stores located in a target region,
- customers who purchased during a campaign,
- students enrolled in courses offered this term.


## Pagila example: customers who made a payment

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customer AS c
WHERE c.customer_id IN (
    SELECT p.customer_id
    FROM payment AS p
)
ORDER BY c.customer_id;
```

The result grain is **one row per customer** because the outer query reads from `customer`.

Even if a customer appears many times in `payment`, membership is simply true or false.


## Compare with a JOIN

A JOIN version might look like:

```sql
SELECT DISTINCT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customer AS c
JOIN payment AS p
    ON c.customer_id = p.customer_id;
```

Why `DISTINCT`?

One customer can have many payment rows, so the JOIN may create multiple rows per customer.

The `IN` version expresses **membership** directly and preserves one outer row per customer.


## When a JOIN may be better

Use a JOIN when you need columns from the related table in the final result.

For example:

```sql
SELECT
    c.first_name,
    c.last_name,
    p.payment_date,
    p.amount
FROM customer AS c
JOIN payment AS p
    ON c.customer_id = p.customer_id;
```

A subquery is not automatically “better.” Choose the form that matches the information you need to return.


# Part 3: Subqueries in `FROM` — Derived Tables


## Treating a query result like a table

A subquery in `FROM` creates a temporary result that the outer query can read.

```sql
SELECT ...
FROM (
    SELECT ...
    FROM ...
) AS derived_table;
```

The subquery must have an alias.


## Example: customer spending totals

First build one row per customer:

```sql
SELECT
    customer_id,
    SUM(amount) AS total_spent
FROM payment
GROUP BY customer_id;
```

Now treat that result as a temporary table:

```sql
SELECT AVG(total_spent) AS avg_customer_spend
FROM (
    SELECT
        customer_id,
        SUM(amount) AS total_spent
    FROM payment
    GROUP BY customer_id
) AS customer_totals;
```


## Why the derived table changes what you can calculate

Inside the subquery:

```text
one row = one customer
```

Outside the subquery:

```text
AVG(total_spent)
```

is therefore the average of **customer totals**, not the average of individual payment rows.

This is a grain change. Understanding the grain of the intermediate result is essential.


## Derived-table alias requirement

This is incomplete:

```sql
SELECT AVG(total_spent)
FROM (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM payment
    GROUP BY customer_id
);
```

Give the derived table a name:

```sql
) AS customer_totals;
```

A good alias describes what the intermediate dataset contains.


## Naming matters

Compare:

```sql
) AS x
```

with:

```sql
) AS customer_totals
```

Both may execute, but the second makes the query easier to reason about.

Treat aliases as short documentation.


# Part 4: Subqueries in `SELECT` and Correlation


## A subquery can produce a column

A scalar subquery may appear in the `SELECT` list.

Example idea:

> Show each film and the number of times it has been rented.

```sql
SELECT
    f.title,
    (
        SELECT COUNT(*)
        FROM inventory AS i
        JOIN rental AS r
            ON i.inventory_id = r.inventory_id
        WHERE i.film_id = f.film_id
    ) AS times_rented
FROM film AS f;
```

This is a **correlated subquery**.


## What makes a subquery correlated?

Inside the subquery appears this reference:

```sql
f.film_id
```

But `f` is defined in the outer query:

```sql
FROM film AS f
```

The inner query depends on the current outer row.

```text
outer film row
     ↓
inner query uses that film_id
     ↓
count related rentals
     ↓
return one value for this film
```


## Non-correlated vs correlated

### Non-correlated

```sql
WHERE length > (SELECT AVG(length) FROM film)
```

The inner query can run independently.

### Correlated

```sql
WHERE i.film_id = f.film_id
```

The inner query needs a value from the current outer row.

This execution difference is conceptually important and can affect performance.


## Why correlated subqueries can be expensive

A correlated subquery may be evaluated repeatedly—conceptually once for each outer row.

If the outer relation contains 1,000 rows, the inner logic may be revisited many times.

That does **not** mean correlated subqueries should never be used. They are often very readable. But on large data, always be aware that a JOIN or pre-aggregation may perform better.


## Equivalent JOIN-and-GROUP-BY idea

A common alternative is:

```sql
SELECT
    f.film_id,
    f.title,
    COUNT(r.rental_id) AS times_rented
FROM film AS f
LEFT JOIN inventory AS i
    ON f.film_id = i.film_id
LEFT JOIN rental AS r
    ON i.inventory_id = r.inventory_id
GROUP BY f.film_id, f.title;
```

This expresses the same type of analysis with joins and aggregation.

The best form depends on readability, correctness, and performance requirements.


## Important difference: preserving zero-match films

Using `LEFT JOIN` preserves films with no rentals.

A correlated `COUNT(*)` over zero matching rows also returns `0`.

An `INNER JOIN` version would eliminate films that have no matching rental.

The structure of the query is therefore part of the business meaning.


# Part 5: `EXISTS` and `NOT EXISTS`


## Sometimes you only need to know whether a match exists

If the analytical question is:

> Does this customer have at least one rental?

then you do not need rental columns or a rental count.

`EXISTS` is designed for this yes/no test.


## `EXISTS` pattern

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customer AS c
WHERE EXISTS (
    SELECT 1
    FROM rental AS r
    WHERE r.customer_id = c.customer_id
);
```

The subquery is correlated because it references `c.customer_id`.


## Why `SELECT 1`?

Inside `EXISTS`, SQL only needs to know whether at least one row is returned.

The actual value selected is irrelevant.

```sql
SELECT 1
```

is a common convention that signals:

> I care about existence, not about returning data from this subquery.


## `NOT EXISTS`: find missing relationships

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customer AS c
WHERE NOT EXISTS (
    SELECT 1
    FROM rental AS r
    WHERE r.customer_id = c.customer_id
);
```

This asks:

> Return customers for whom no matching rental row exists.


## Real-world `NOT EXISTS` questions

The pattern appears in many domains:

- customers with no purchase in the last 90 days,
- employees with no submitted timesheet,
- orders with no payment,
- patients with no follow-up appointment,
- students with no assignment submission,
- products with no inventory record.

This is the same analytical idea as the anti-join pattern from Module 4, expressed with a subquery.


# Part 6: The `NOT IN` and `NULL` Trap


## Why `NOT IN` requires care

Suppose you write:

```sql
WHERE customer_id NOT IN (
    SELECT customer_id
    FROM some_table
)
```

If the inner result contains a `NULL`, SQL enters three-valued logic: true, false, and unknown.

A single unknown value can prevent `NOT IN` from returning the rows you expect.


## Safer alternatives

When testing for nonexistence, prefer:

```sql
WHERE NOT EXISTS (...)
```

or explicitly remove nulls from the inner result:

```sql
WHERE customer_id NOT IN (
    SELECT customer_id
    FROM some_table
    WHERE customer_id IS NOT NULL
)
```

`NOT EXISTS` usually communicates the intent more directly.


## Quick check

Which statement is easier to trust when the inner column may contain `NULL`?

```sql
NOT IN (subquery)
```

or

```sql
NOT EXISTS (correlated_subquery)
```

For a missing-match question, `NOT EXISTS` is generally the safer and clearer choice.


# Part 7: Common Table Expressions — Naming Intermediate Results


## What is a CTE?

A **common table expression**, or **CTE**, is a named query result defined with `WITH`.

Basic form:

```sql
WITH cte_name AS (
    SELECT ...
)
SELECT ...
FROM cte_name;
```

The name exists for the duration of that statement.


## From nested subquery to named step

Subquery form:

```sql
SELECT customer_id, total_spent
FROM (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM payment
    GROUP BY customer_id
) AS customer_totals
WHERE total_spent > 150;
```

CTE form:

```sql
WITH customer_totals AS (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM payment
    GROUP BY customer_id
)
SELECT customer_id, total_spent
FROM customer_totals
WHERE total_spent > 150;
```


## Same logic, different layout

The two versions can express the same logic.

The CTE moves the intermediate result to the top and gives it a meaningful name.

```text
WITH customer_totals AS (...)
              ↓
      SELECT FROM customer_totals
```

For one short step, the difference is small. For many steps, readability improves substantially.


## Think of a CTE as a named stage

A useful mental model:

```text
raw rows
  ↓
customer_totals
  ↓
overall_average
  ↓
above_average
  ↓
final report
```

Each stage should have:

- a clear purpose,
- a clear grain,
- a clear name.


## Chaining CTEs

Multiple CTEs are separated by commas.

```sql
WITH step_1 AS (
    SELECT ...
),
step_2 AS (
    SELECT ...
    FROM step_1
),
step_3 AS (
    SELECT ...
    FROM step_2
)
SELECT ...
FROM step_3;
```

A later CTE can use a CTE defined earlier.


# Part 8: A Multi-Step CTE Example


## Business question

> Which customers have spent more than the average customer, and how much did they spend?

Break the question into steps:

1. total payments per customer,
2. average of those customer totals,
3. keep customers above the average,
4. attach customer names.


## Step 1: total spend per customer

```sql
WITH customer_totals AS (
    SELECT
        customer_id,
        SUM(amount) AS total_spent
    FROM payment
    GROUP BY customer_id
)
```

Grain:

```text
one row per customer
```


## Step 2: overall average

Add another CTE:

```sql
WITH customer_totals AS (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM payment
    GROUP BY customer_id
),
overall_average AS (
    SELECT AVG(total_spent) AS avg_spent
    FROM customer_totals
)
```

The second step returns exactly one row.


## Step 3: filter above-average customers

```sql
above_average AS (
    SELECT
        ct.customer_id,
        ct.total_spent
    FROM customer_totals AS ct
    CROSS JOIN overall_average AS oa
    WHERE ct.total_spent > oa.avg_spent
)
```

Because `overall_average` has one row, the cross join simply makes the single average available to each customer-total row.


## Why this `CROSS JOIN` is safe

In Module 4, cross joins were dangerous when both sides had many rows.

Here:

```text
customer_totals: many rows
overall_average: exactly 1 row
```

So the row count remains:

```text
N × 1 = N
```

The lesson is not “cross joins are bad.” The lesson is to understand the cardinality before using one.


## Step 4: final result

```sql
WITH customer_totals AS (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM payment
    GROUP BY customer_id
),
overall_average AS (
    SELECT AVG(total_spent) AS avg_spent
    FROM customer_totals
),
above_average AS (
    SELECT ct.customer_id, ct.total_spent
    FROM customer_totals AS ct
    CROSS JOIN overall_average AS oa
    WHERE ct.total_spent > oa.avg_spent
)
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    aa.total_spent
FROM above_average AS aa
JOIN customer AS c
    ON aa.customer_id = c.customer_id
ORDER BY aa.total_spent DESC;
```


## Read the query as a story

```text
customer_totals
    "What did each customer spend?"
        ↓
overall_average
    "What is the average customer total?"
        ↓
above_average
    "Who is above that average?"
        ↓
final SELECT
    "Add customer names and sort."
```

A well-designed CTE query should be understandable from the step names before every SQL detail is inspected.


# Part 9: Debugging CTE Pipelines


## Test one stage at a time

If the final result is wrong, do not debug the entire query at once.

Temporarily replace the final query:

```sql
SELECT *
FROM customer_totals
ORDER BY total_spent DESC;
```

Then inspect the next stage:

```sql
SELECT *
FROM overall_average;
```

Then:

```sql
SELECT *
FROM above_average;
```


## Validate the grain of every CTE

Ask of every step:

> What does one row represent here?

Examples:

| CTE | Expected grain |
|---|---|
| `customer_totals` | one row per customer |
| `overall_average` | one row total |
| `above_average` | one row per qualifying customer |

Unexpected duplicates often reveal an incorrect join or grouping decision.


## Validate row counts

Useful checks include:

```sql
SELECT COUNT(*) FROM customer_totals;
```

and:

```sql
SELECT COUNT(DISTINCT customer_id)
FROM customer_totals;
```

If a result that should be one row per customer has many more rows than distinct customer IDs, investigate the logic.


## Validate key uniqueness

Suppose `above_average` should contain one row per customer.

```sql
SELECT
    customer_id,
    COUNT(*)
FROM above_average
GROUP BY customer_id
HAVING COUNT(*) > 1;
```

If this returns rows, the assumed grain has been violated.


## CTE names should describe results

Weak names:

```text
cte1
cte2
temp
data
```

Stronger names:

```text
customer_totals
overall_average
above_average
monthly_revenue
```

A query is easier to maintain when the names explain what each step produces.


# Part 10: Set Operations — Combining Results Vertically


## JOINs and set operations solve different shapes

A JOIN usually combines columns **horizontally**.

```text
customer columns + rental columns
```

A set operation combines rows **vertically**.

```text
query 1 rows
query 2 rows
query 3 rows
```

This distinction is fundamental.


## The four operations in this module

PostgreSQL supports:

- `UNION`
- `UNION ALL`
- `INTERSECT`
- `EXCEPT`

Each operator combines the complete results of two compatible queries.


## Compatibility rule 1: same number of columns

This works structurally:

```sql
SELECT first_name, last_name FROM customer
UNION ALL
SELECT first_name, last_name FROM actor;
```

Both queries return two columns.

This does not:

```sql
SELECT first_name, last_name FROM customer
UNION ALL
SELECT first_name FROM actor;
```

The column counts do not match.


## Compatibility rule 2: corresponding types must be compatible

Column positions are aligned by position, not by meaning.

```text
query 1 column 1 ↔ query 2 column 1
query 1 column 2 ↔ query 2 column 2
```

Make sure corresponding expressions can be reconciled to compatible PostgreSQL types.


## Column names come from the first query

```sql
SELECT first_name AS given_name,
       last_name  AS family_name
FROM customer
UNION ALL
SELECT first_name,
       last_name
FROM actor;
```

The combined output uses the column names established by the first `SELECT`.


# Part 11: `UNION ALL` and `UNION`


## `UNION ALL`: stack everything

```sql
SELECT first_name, last_name
FROM customer
UNION ALL
SELECT first_name, last_name
FROM actor;
```

`UNION ALL` preserves every row from both results, including duplicates.


## `UNION`: stack and deduplicate

```sql
SELECT first_name, last_name
FROM customer
UNION
SELECT first_name, last_name
FROM actor;
```

`UNION` removes duplicate full rows from the combined result.

That extra deduplication has a cost.


## Which should be the default?

Use `UNION ALL` when duplicates are meaningful or when you simply want to append datasets.

Use `UNION` when the business question requires a distinct combined set.

Do not use `UNION` merely because it “looks safer.” Removing duplicates can hide information.


## Real-world example: combine event feeds

Suppose one system stores web purchases and another stores store purchases with the same output structure.

```sql
SELECT customer_id, purchase_date, amount, 'web' AS channel
FROM web_sales
UNION ALL
SELECT customer_id, purchase_date, amount, 'store' AS channel
FROM store_sales;
```

Keeping all rows is usually correct because two identical-looking purchases may still be separate events.


## Real-world example: distinct contact list

If the goal is a unique list of email addresses from two campaign sources:

```sql
SELECT email FROM campaign_a
UNION
SELECT email FROM campaign_b;
```

Here deduplication is part of the requirement.


# Part 12: `INTERSECT`


## `INTERSECT`: rows present in both results

```sql
SELECT first_name, last_name
FROM customer
INTERSECT
SELECT first_name, last_name
FROM actor;
```

This returns full rows that appear in both result sets.


## Real-world `INTERSECT` questions

Examples:

- customers appearing in both campaign lists,
- products sold in both regions,
- employees certified in both required skill sets,
- users active in both January and February.

`INTERSECT` makes “common to both sets” explicit.


## Whole-row comparison matters

If you write:

```sql
SELECT first_name, last_name
...
INTERSECT
SELECT first_name, last_name
...
```

both columns together define the row being compared.

If the first name matches but the last name does not, the row is not the same.


# Part 13: `EXCEPT`


## `EXCEPT`: rows in the first result but not the second

```sql
SELECT first_name, last_name
FROM actor
EXCEPT
SELECT first_name, last_name
FROM customer;
```

This asks:

> Which actor-name rows do not appear in the customer-name result?


## Order matters

These are different questions:

```sql
A EXCEPT B
```

and

```sql
B EXCEPT A
```

The first asks what is in A but not B. The second asks what is in B but not A.


## Real-world `EXCEPT` questions

- active customers not in the loyalty program,
- expected records not found in the received file,
- employees on the roster but not in the payroll export,
- products in the catalog but not in current inventory.

This makes `EXCEPT` useful for reconciliation and data-quality work.


# Part 14: Ordering Set-Operation Results


## One `ORDER BY` at the end

To sort the complete combined result, place `ORDER BY` after the final query:

```sql
SELECT first_name, last_name
FROM customer
UNION ALL
SELECT first_name, last_name
FROM actor
ORDER BY last_name, first_name;
```

The ordering applies to the combined result.


## Why internal ordering is usually meaningless here

Set operations work with result sets. Unless a subquery needs ordering for a specific operation such as `LIMIT`, the order of rows from each input query is not something to rely on.

Think:

```text
build result set A
build result set B
combine them
then order the final result
```


# Part 15: Choosing the Right Tool


## Four tools, four common intentions

| Need | Usually start with |
|---|---|
| Add columns from related records | `JOIN` |
| Filter using a computed value or list | subquery |
| Express multi-step logic as named stages | CTE |
| Stack or compare complete result sets | set operation |

These are starting points, not rigid laws.


## JOIN vs subquery

Question:

> Which customers rented ACADEMY DINOSAUR?

A JOIN works well if you also need rental or film columns.

A subquery works well if you only need to filter the customer population.

Choose the structure that states the intent with the least unnecessary complexity.


## Subquery vs CTE

For one short nested step:

```sql
WHERE amount > (SELECT AVG(amount) FROM payment)
```

is compact and clear.

For several dependent transformations, CTEs usually improve readability because each step can be named and inspected.


## CTE vs view

A CTE exists only within one SQL statement.

A view stores a query definition so it can be reused by later statements.

Views are introduced in the textbook alongside subqueries and CTEs, but Module 5 focuses primarily on subqueries, CTEs, and set operations.


## Set operation vs JOIN

Ask whether you are combining:

```text
columns beside each other  → JOIN
```

or

```text
rows under each other      → set operation
```

This simple distinction prevents many design mistakes.


# Part 16: Worked Pagila Examples


## Example 1: films above average length

```sql
SELECT
    film_id,
    title,
    length
FROM film
WHERE length > (
    SELECT AVG(length)
    FROM film
)
ORDER BY length DESC, title;
```

### Before you run

Predict:

- Does the inner query return one row or many?
- Can `>` compare `length` to that result?
- What does one output row represent?


## Example 2: customers with at least one payment

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customer AS c
WHERE EXISTS (
    SELECT 1
    FROM payment AS p
    WHERE p.customer_id = c.customer_id
)
ORDER BY c.customer_id;
```

### Interpretation

The outer query controls the result grain: one row per customer.


## Example 3: customers with no payment

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customer AS c
WHERE NOT EXISTS (
    SELECT 1
    FROM payment AS p
    WHERE p.customer_id = c.customer_id
)
ORDER BY c.customer_id;
```

This is a missing-match analysis expressed with `NOT EXISTS`.


## Example 4: average customer spending

```sql
SELECT AVG(total_spent) AS avg_customer_spend
FROM (
    SELECT
        customer_id,
        SUM(amount) AS total_spent
    FROM payment
    GROUP BY customer_id
) AS customer_totals;
```

### Grain trace

```text
payment table        → one row per payment
customer_totals      → one row per customer
outer AVG            → one row total
```


## Example 5: rental counts with a CTE

```sql
WITH rental_counts AS (
    SELECT
        customer_id,
        COUNT(*) AS rental_count
    FROM rental
    GROUP BY customer_id
)
SELECT
    customer_id,
    rental_count
FROM rental_counts
ORDER BY rental_count DESC, customer_id;
```

The CTE names the intermediate dataset and keeps the final `SELECT` simple.


## Example 6: customers above average rental count

```sql
WITH rental_counts AS (
    SELECT
        customer_id,
        COUNT(*) AS rental_count
    FROM rental
    GROUP BY customer_id
),
avg_rental_count AS (
    SELECT AVG(rental_count) AS avg_count
    FROM rental_counts
)
SELECT
    rc.customer_id,
    rc.rental_count
FROM rental_counts AS rc
CROSS JOIN avg_rental_count AS a
WHERE rc.rental_count > a.avg_count
ORDER BY rc.rental_count DESC;
```


## Example 7: combine customer and actor names

```sql
SELECT first_name, last_name, 'customer' AS source
FROM customer
UNION ALL
SELECT first_name, last_name, 'actor' AS source
FROM actor
ORDER BY last_name, first_name, source;
```

Adding a source label helps preserve the origin of each row.


## Example 8: names found in both tables

```sql
SELECT first_name, last_name
FROM customer
INTERSECT
SELECT first_name, last_name
FROM actor
ORDER BY last_name, first_name;
```

The query compares complete `(first_name, last_name)` rows.


# Part 17: Common Errors and How to Diagnose Them


## Error 1: scalar subquery returns multiple rows

Problem:

```sql
WHERE rental_rate = (
    SELECT rental_rate
    FROM film
)
```

Diagnosis:

The outer operator expects one value, but the subquery returns many.

Possible fix: use an aggregate, add a filter that guarantees one row, or use `IN` if a list is intended.


## Error 2: derived table has no alias

Problem:

```sql
SELECT *
FROM (
    SELECT customer_id
    FROM payment
);
```

Fix:

```sql
) AS payment_customers;
```


## Error 3: `NOT IN` behaves unexpectedly

Symptom:

The query returns no rows even though unmatched records appear to exist.

Check whether the inner query can return `NULL`.

Prefer `NOT EXISTS` for a missing-match test.


## Error 4: CTE grain is not what you assumed

Symptom:

The final result contains unexpected duplicates or inflated counts.

Diagnostic questions:

- What should one row represent in this CTE?
- Is the grouping key correct?
- Did a JOIN create one-to-many fan-out?
- Are you aggregating before or after the fan-out?


## Error 5: set-operation column counts differ

Problem:

```sql
SELECT first_name, last_name
FROM customer
UNION ALL
SELECT first_name
FROM actor;
```

Both branches must return the same number of columns.


## Error 6: set-operation types do not align

Even when the column counts match, corresponding positions must have compatible data types.

If necessary, use an explicit cast so the intended common type is clear.

```sql
SELECT customer_id::text AS identifier
FROM customer
UNION ALL
SELECT staff_id::text
FROM staff;
```


## Error 7: using `UNION` when duplicates matter

If the two inputs each contain a valid identical row, `UNION` may collapse them into one row.

Ask:

> Is a duplicate here an error, or is it a legitimate event from another source?

If legitimate rows must be preserved, use `UNION ALL`.


## Error 8: `EXCEPT` branches reversed

```sql
A EXCEPT B
```

is directional.

If the result looks backward, verify which dataset should define the population of interest.


# Part 18: Guided Practice


## Practice 1: scalar subquery

Return each film whose `rental_rate` is greater than the average rental rate across all films.

Include:

- `film_id`
- `title`
- `rental_rate`

Sort highest rate first.

### Hint

The inner query should return exactly one value.


## Practice 1 solution

```sql
SELECT
    film_id,
    title,
    rental_rate
FROM film
WHERE rental_rate > (
    SELECT AVG(rental_rate)
    FROM film
)
ORDER BY rental_rate DESC, title;
```


## Practice 2: `IN` subquery

Return customers who have made at least one payment.

Do not use a JOIN.

### Hint

The inner query can return one column containing many customer IDs.


## Practice 2 solution

```sql
SELECT
    customer_id,
    first_name,
    last_name
FROM customer
WHERE customer_id IN (
    SELECT customer_id
    FROM payment
)
ORDER BY customer_id;
```


## Practice 3: `NOT EXISTS`

Return customers who have never made a payment.

Then explain why `NOT EXISTS` is safer than `NOT IN` if the inner key could contain `NULL`.


## Practice 3 solution

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customer AS c
WHERE NOT EXISTS (
    SELECT 1
    FROM payment AS p
    WHERE p.customer_id = c.customer_id
)
ORDER BY c.customer_id;
```

`NOT EXISTS` tests for the absence of a matching row and is not invalidated by an unrelated `NULL` in the same way `NOT IN` can be.


## Practice 4: derived table

Compute the average total amount paid per customer.

You need two levels:

1. sum payments for each customer,
2. average those totals.


## Practice 4 solution

```sql
SELECT AVG(total_spent) AS avg_customer_spend
FROM (
    SELECT
        customer_id,
        SUM(amount) AS total_spent
    FROM payment
    GROUP BY customer_id
) AS customer_totals;
```


## Practice 5: CTE chain

Using CTEs:

1. count rentals per customer,
2. compute the average rental count,
3. keep customers above that average.

Name the steps descriptively.


## Practice 5 solution

```sql
WITH rental_counts AS (
    SELECT customer_id, COUNT(*) AS rental_count
    FROM rental
    GROUP BY customer_id
),
avg_rental_count AS (
    SELECT AVG(rental_count) AS avg_count
    FROM rental_counts
)
SELECT
    rc.customer_id,
    rc.rental_count
FROM rental_counts AS rc
CROSS JOIN avg_rental_count AS a
WHERE rc.rental_count > a.avg_count
ORDER BY rc.rental_count DESC;
```


## Practice 6: `UNION ALL`

Create one list of names from `customer` and `actor`.

Add a third column called `source` showing whether the row came from `customer` or `actor`.

Keep duplicates.


## Practice 6 solution

```sql
SELECT
    first_name,
    last_name,
    'customer' AS source
FROM customer
UNION ALL
SELECT
    first_name,
    last_name,
    'actor' AS source
FROM actor
ORDER BY last_name, first_name, source;
```


## Practice 7: `INTERSECT`

Find `(first_name, last_name)` pairs that appear in both `customer` and `actor`.


## Practice 7 solution

```sql
SELECT first_name, last_name
FROM customer
INTERSECT
SELECT first_name, last_name
FROM actor
ORDER BY last_name, first_name;
```


## Practice 8: `EXCEPT`

Find customer name pairs that do not appear in the actor table.

Then reverse the query and explain why the meaning changes.


## Practice 8 solution

```sql
SELECT first_name, last_name
FROM customer
EXCEPT
SELECT first_name, last_name
FROM actor
ORDER BY last_name, first_name;
```

Reversing the branches asks for actor names that do not appear among customers. `EXCEPT` is directional.


# Part 19: AI Critique Practice


## Critique this query

A colleague asks for customers who have never made a payment and produces:

```sql
SELECT customer_id, first_name, last_name
FROM customer
WHERE customer_id NOT IN (
    SELECT customer_id
    FROM payment
);
```

### Questions

1. What assumption must be true for this to behave safely?
2. What alternative expresses missing-match logic more robustly?
3. What result checks would you run before trusting the answer?


## Critique response

The query assumes the subquery cannot return `NULL` for `customer_id`. If a `NULL` can appear, `NOT IN` can produce an unexpected empty result because comparisons against unknown values are not true.

A safer form is:

```sql
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customer AS c
WHERE NOT EXISTS (
    SELECT 1
    FROM payment AS p
    WHERE p.customer_id = c.customer_id
);
```

Validation should include row counts, spot checks of returned customer IDs, and a comparison with a `LEFT JOIN ... WHERE p.customer_id IS NULL` anti-join.


## Critique this CTE chain

```sql
WITH a AS (
    SELECT customer_id, SUM(amount) AS x
    FROM payment
    GROUP BY customer_id
),
b AS (
    SELECT AVG(x) AS y FROM a
),
c AS (
    SELECT a.customer_id, a.x
    FROM a, b
    WHERE a.x > b.y
)
SELECT * FROM c;
```

The logic may be valid, but what makes it hard to maintain?


## Improve the CTE names

```sql
WITH customer_totals AS (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM payment
    GROUP BY customer_id
),
overall_average AS (
    SELECT AVG(total_spent) AS avg_spent
    FROM customer_totals
),
above_average AS (
    SELECT ct.customer_id, ct.total_spent
    FROM customer_totals AS ct
    CROSS JOIN overall_average AS oa
    WHERE ct.total_spent > oa.avg_spent
)
SELECT *
FROM above_average;
```

The revised names make the analytical steps visible before every expression is inspected.


# Part 20: Knowledge Check


## Knowledge check 1

Which subquery shape is required here?

```sql
WHERE length > ( ... )
```

A. many rows and many columns  
B. one scalar value  
C. one column with many values  
D. any shape

**Answer: B.** A comparison such as `>` expects one value on the right-hand side.


## Knowledge check 2

Which operator is most natural when the inner query returns a list of IDs?

A. `IN`  
B. `>`  
C. `ORDER BY`  
D. `GROUP BY`

**Answer: A.** `IN` checks whether the outer value belongs to a list returned by the subquery.


## Knowledge check 3

What makes a subquery correlated?

A. it contains an aggregate function  
B. it references a column from the outer query  
C. it is placed in `FROM`  
D. it contains `ORDER BY`

**Answer: B.** A correlated subquery depends on the current outer row.


## Knowledge check 4

Which is generally safest for “rows with no matching record” when nulls may be present?

A. `NOT EXISTS`  
B. `NOT IN` without checking nulls  
C. `CROSS JOIN`  
D. `UNION`

**Answer: A.** `NOT EXISTS` directly tests whether a related row is absent.


## Knowledge check 5

What is the primary readability advantage of a CTE?

A. it automatically creates an index  
B. it permanently stores data  
C. it gives an intermediate result a meaningful name  
D. it always runs faster than a subquery

**Answer: C.** CTEs make multi-step logic easier to read and test. They are not automatically faster.


## Knowledge check 6

Which operation keeps duplicates by default?

A. `UNION`  
B. `UNION ALL`  
C. `INTERSECT`  
D. `EXCEPT`

**Answer: B.** `UNION ALL` appends all rows without deduplication.


## Knowledge check 7

What must be true for the two branches of a set operation?

A. they must use the same table  
B. they must contain identical `WHERE` clauses  
C. they must return the same number of columns with compatible types  
D. they must have the same aliases

**Answer: C.** Set operations align outputs by column position.


## Knowledge check 8

What does `A EXCEPT B` return?

A. rows in both A and B  
B. all rows from A and B  
C. rows in A that are not in B  
D. rows in B that are not in A

**Answer: C.** `EXCEPT` is directional.


# Part 21: Concept Map


## Module 5 concept map

```text
                         DERIVED DATASETS
                               |
          +--------------------+--------------------+
          |                    |                    |
      SUBQUERIES              CTEs            SET OPERATIONS
          |                    |                    |
   +------+------+       named steps       +--------+---------+
   |             |            |             |        |         |
 scalar       list/table   WITH ... AS     UNION   INTERSECT  EXCEPT
   |             |            |             |
 comparisons    IN/FROM   chain stages    UNION ALL
   |
 correlated?
   |
 EXISTS / NOT EXISTS

JOIN remains the natural tool when the goal is to add related columns horizontally.
```


# Part 22: Lab 4 Readiness


## Before starting Lab 4

You should be able to complete each of these without copying an example:

- write a scalar subquery in `WHERE`
- write an `IN` subquery that returns one column
- write a derived table in `FROM` and give it an alias
- explain what makes a subquery correlated
- write `EXISTS` and `NOT EXISTS`
- build one CTE and then chain multiple CTEs
- identify the grain of each intermediate result
- write all four set operations
- explain when duplicate removal is appropriate
- explain why `EXCEPT` order matters


## Lab mindset

For every query, verify four things:

1. **Business meaning** — Does the SQL answer the question actually asked?
2. **Result grain** — What does one row represent?
3. **Row preservation** — Which records can disappear or duplicate?
4. **Validation** — What row-count or spot-check query would increase your confidence?

Correct syntax is only the first layer of a trustworthy analytical query.


# Part 23: Discussion Preparation


## Breaking a complex question into steps

When an analytical question contains several dependent calculations, do not begin by nesting everything immediately.

Write the steps in plain language first.

Example:

> Find customers whose total spending is above the average total spending of customers in their store.

Possible stages:

1. calculate customer totals,
2. calculate store-level average customer totals,
3. compare each customer with the relevant store average,
4. attach customer names and store information.


## Questions to ask about your own query structure

- Which intermediate result deserves a name?
- Which step changes the grain?
- Which step should be tested first?
- Could a short scalar subquery be clearer than another CTE?
- Are two complete result sets being stacked or compared?
- Will duplicates represent real events or redundant rows?
- Could another analyst understand the logic from the CTE names alone?


# Part 24: Key Takeaways


## Key takeaways

- A `SELECT` result can be treated as input to another query.
- A scalar subquery returns one value; a list-returning subquery often pairs with `IN`; a `FROM` subquery behaves like a derived table.
- Correlated subqueries depend on values from the outer row.
- `EXISTS` and `NOT EXISTS` are natural tools for presence and absence questions.
- Be cautious with `NOT IN` when the inner result may contain `NULL`.
- CTEs name intermediate results and make multi-step logic easier to read, test, and review.
- `UNION ALL` preserves duplicates; `UNION` removes them.
- `INTERSECT` finds rows common to both results; `EXCEPT` finds rows in the first result but not the second.
- Set-operation branches need matching column counts and compatible types.
- Choose the SQL form that communicates the analytical intent most clearly.


## One final decision guide

```text
Need related columns beside each other?
    → JOIN

Need to filter using a computed value or computed list?
    → subquery

Need several named analytical stages?
    → CTE

Need to append or compare complete row sets?
    → UNION / UNION ALL / INTERSECT / EXCEPT
```

More than one approach may be correct. Readability and result validation are part of correctness.


# References


## References and supporting resources

**Principal text**

Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for Data Analytics: Analyze Data Effectively, Uncover Insights and Master Advanced SQL for Real-World Applications* (4th ed.). Packt Publishing. Chapter 7, “Defining Datasets from Existing Datasets.”

**PostgreSQL documentation**

- PostgreSQL Global Development Group. *WITH Queries (Common Table Expressions)*, PostgreSQL 16 documentation.
- PostgreSQL Global Development Group. *Subquery Expressions*, PostgreSQL 16 documentation.
- PostgreSQL Global Development Group. *Combining Queries (UNION, INTERSECT, EXCEPT)*, PostgreSQL 16 documentation.

**Practice database**

- Pagila sample PostgreSQL database used throughout ALY 6420.
